### Read the product data from the Bronze layer, apply the required cleaning and standardization steps, then save the final cleaned output to the Silver layer as a Delta table.

# Initialization


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim, length
from pyspark.sql.window import Window

# Read Bronze table


In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

# Display the raw Bronze data first


In [0]:
# Display the raw Bronze data first to understand the file structure
# and identify the data quality issues that need cleaning in the Silver layer.
df.display()

- ============================================================

Data quality issues found in this file:
 1. Some string columns may contain leading or trailing spaces.
 2. The prd_cost column contains some null values.
 3. The prd_line column contains short code values and some null values.
 4. Date columns need to be converted to proper date format.
 5. Some end dates are earlier than start dates and should be cleaned.
 6. The product key needs to be standardized to match the sales file in future joins.
 7. Some product records are duplicated based on the product key, so we should keep the latest version.
 8. Column names need to be renamed to cleaner business-friendly names before saving the final Silver table.

============================================================

# Silver Transformations


# Trimming

In [0]:
# Trimming string columns
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# handle null values & invalid data in coulmns


In [0]:
# Validation checks before cleaning product data
df.select(
    F.count(F.when(col("prd_cost").isNull(), True)).alias("null_product_cost_count"),
    F.count(F.when(col("prd_line").isNull(), True)).alias("null_product_line_count"),
    F.count(
        F.when(
            F.to_date(col("prd_end_dt"), "yyyy-MM-dd").isNotNull() &
            (F.to_date(col("prd_end_dt"), "yyyy-MM-dd") < F.to_date(col("prd_start_dt"), "yyyy-MM-dd")),
            True
        )
    ).alias("invalid_date_range_count")
).display()

# Cleaning Dates



In [0]:
# Converting date columns to proper date format
df = (
    df
    .withColumn("prd_start_dt", F.to_date(col("prd_start_dt"), "yyyy-MM-dd"))
    .withColumn("prd_end_dt", F.to_date(col("prd_end_dt"), "yyyy-MM-dd"))
)

# If end date is earlier than start date, set it to null
df = df.withColumn(
    "prd_end_dt",
    F.when(col("prd_end_dt") < col("prd_start_dt"), None)
     .otherwise(col("prd_end_dt"))
)

##  Validation checks after cleaning dates


In [0]:
# Validation checks after cleaning product dates
df.select(
    F.count(
        F.when(
            col("prd_end_dt").isNotNull() & (col("prd_end_dt") < col("prd_start_dt")),
            True
        )
    ).alias("invalid_date_range_count")
).display()

# Standardizing product line values



In [0]:
# Standardizing product line values
df = df.withColumn(
    "prd_line",
    F.when(col("prd_line") == "M", "Mountain")
     .when(col("prd_line") == "R", "Road")
     .when(col("prd_line") == "S", "Other Sales")
     .when(col("prd_line") == "T", "Touring")
     .otherwise("n/a")
)

##Validation checks after standardizing product line


In [0]:
# Validation checks after standardizing product line values
df.select(
    F.count(F.when(col("prd_line").isNull(), True)).alias("null_product_line_count"),
    F.count(F.when(col("prd_line") == "n/a", True)).alias("unknown_product_line_count")
).display()

#cost cleanup

In [0]:
#Replacing null product cost values with 0
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

# Product Key Parsing


In [0]:
# Extracting category ID from the product key
# and standardizing the product key to match the sales file
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

In [0]:
df.display()

# check duplicate record on bussiness keys in this file

In [0]:
# Check for duplicate records based on the business key
# to make sure the product rows are unique before saving to Silver.
df.groupBy("prd_key") \
  .count() \
  .filter(col("count") > 1) \
  .display()

## Removing duplicates


In [0]:
# Removing duplicate product records by keeping the latest start date
window_spec = Window.partitionBy("prd_key").orderBy(col("prd_start_dt").desc())

df = (
    df
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

##  Validation after removing duplicates


In [0]:
# Validation checks after removing duplicate product records
df.groupBy("prd_key") \
  .count() \
  .filter(col("count") > 1) \
  .count()

#rename coulmns


In [0]:
# Renaming columns to cleaner business names
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

#quick validation for df


In [0]:
#Display the cleaned dataframe for a quick validation check
df.display()


#writing silver table


In [0]:
# Writing Silver table
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

#check table created or not


In [0]:
%sql
-- Preview the final Silver table
SELECT * FROM workspace.silver.crm_products LIMIT 10